# Configuration

In [1]:
import logging
import time
from datetime import datetime
import warnings
import shutil
import json
import pickle
import torch
import sys

import os
import pandas as pd
import numpy as np
import networkx as nx
import scanpy as sc
import anndata as ad

import torch.nn as nn
from tqdm import tqdm
from collections import Counter
from itertools import islice

from torch.cuda.amp import autocast, GradScaler  # 新增：混合精度训练
import psutil  # 新增：系统监控

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

sys.path.append("/home/liyang/BioWuYan/dygmamba_project/model/dygmamba/src")

# 1. 加载 autoreload 扩展
%load_ext autoreload

# 2. 设置模式为 "2" (表示自动重载所有模块)
%autoreload 2


from models.DyGMamba import DyGMamba
from models.modules import MergeLayer, MergeLayerTD

from utils.load_configs import load_link_prediction_args

from utils.DataLoader import get_model_data
from utils.DataLoader import get_idx_data_loader
from utils.utils import get_neighbor_sampler, NegativeEdgeSampler
from utils.utils import get_parameter_sizes
from utils.utils import set_random_seed
from utils.utils import convert_to_gpu, create_optimizer
from utils.EarlyStopping import EarlyStopping
from utils.metrics import get_link_prediction_metrics
from models.evaluate_models_utils import evaluate_model_link_prediction
from models.inference_grn import model_link_prediction

In [2]:
cell_type = "HepG2"

print("********************** start ********************")

start_time = time.time()  # start the time
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Start the job")

# get arguments
args = load_link_prediction_args(is_evaluation=False)

# ============ 新增：优化配置参数 ============
USE_AMP = True  # 混合精度训练开关
ACCUMULATION_STEPS = 4  # 梯度累积步数
MAX_NEIGHBORS = 30  # 最大邻居采样数

print("**********************device********************")
print(f"Now use device is {args.device}")


********************** start ********************
[2026-02-12 16:53:34] Start the job
**********************device********************
Now use device is cuda:0


# Data Read

In [ ]:
#########################################################################################
# *************************** load data ******************************************

data_path = "/home/liyang/BioWuYan/dygmamba_project/data/cell_line/" + cell_type + "/data_dyg2/"

feat_path = data_path + "edge_features.npy"
edge_label_path = data_path + "edge_labels.npy"

Edge_feature = np.load(feat_path, mmap_mode="r")
Edge_feature = Edge_feature.reshape(-1,1).copy()
Edge_label = np.load(edge_label_path, mmap_mode="r")
Edge_label = Edge_label.reshape(-1,1).copy()

with open(data_path + "node_feature_data.pkl", "rb") as f:
    load_data = pickle.load(f)

Node_feature = load_data['node_feature']

Node_id = pd.read_pickle(data_path + "node_id.pkl")

graph_df = pd.read_pickle(data_path + "Graph_df.pkl")
graph_df["Unnamed"] = graph_df.index
name_list = ["Unnamed", "source_node", "target_node", "time", "label", "edge_idx"]
New_Graph = graph_df[name_list].copy()
New_Graph.columns = ['Unnamed: 0', 'u', 'i', 'ts', 'label', 'idx']

print("********************** Successfully load data ********************")

#########################################################################################
# *************************** process data ******************************************

node_raw_feature_dict, edge_raw_features, full_data= get_model_data(New_Graph,
                                                        Edge_feature,
                                                        Node_feature,
                                                        feature_dim= 172)


edge_raw_features = torch.from_numpy(edge_raw_features.astype(np.float32)).to(args.device)
if torch.isnan(edge_raw_features).any():
    print(f"Edge_feature has Nan values. Please check the data.")
else:
    print(f"Edge_feature is successfully converted to tensor without NaN values.")
    
Edge_label = torch.from_numpy(Edge_label.astype(np.float32)).to(args.device)

print("********************** Successfully process data ********************")


********************** Successfully load data ********************
The dataset has 6481612 interactions, involving 33516 different nodes
********************** Successfully process data ********************


In [ ]:
print(len(node_raw_feature_dict.keys()))

print(list(node_raw_feature_dict.keys())[0:10])

has_nan = np.isnan(list(node_raw_feature_dict.values())).any()
print(has_nan)

print(list(node_raw_feature_dict.values())[0])

print(type(next(iter(node_raw_feature_dict.values()))))

14965530
[(1, 0.0), (1, 0.0043132993656562), (1, 0.0099594085638523), (1, 0.0239404208482221), (1, 0.0535768513363072), (1, 0.0653843180380873), (1, 0.0661062260727917), (1, 0.174704324580689), (1, 0.206456884792548), (1, 0.209561770213248)]
False
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0.]


<class 'numpy.ndarray'>


In [ ]:
print(len(edge_raw_features))

print(edge_raw_features[0:10])



## data size

In [ ]:
print(f"node feature shape: {len(node_raw_feature_dict.keys())},\n \
    edge feature shape: {edge_raw_features.shape},\n \
    edge label shape: {Edge_label.shape}\n")

In [14]:
##########################################################
# get neighbor data

full_neighbor_sampler = get_neighbor_sampler(data=full_data,
                                            sample_neighbor_strategy=args.sample_neighbor_strategy,
                                            time_scaling_factor=args.time_scaling_factor, 
                                            seed=1,
                                            max_neighbors=MAX_NEIGHBORS) # 新增：限制邻居数量
print("********************* successfully get neighbor **************")
neg_edge_sampler = NegativeEdgeSampler(src_node_ids=full_data.src_node_ids,
                                            dst_node_ids=full_data.dst_node_ids,
                                            interact_times=full_data.node_interact_times,
                                            negative_sample_strategy='inductive',
                                            seed=2)
print("********************* successfully get negative neighbor **************")
###########################################################
# get index data for batch analysis
effective_batch_size = args.batch_size * ACCUMULATION_STEPS

idx_data_loader = get_idx_data_loader(indices_list=list(range(len(full_data.src_node_ids))),
                                        batch_size=effective_batch_size,  # 修改：使用更大的batch size
                                        shuffle=True)  # 修改：启用数据打乱


metric_all_runs = []

print("********************** Successfully process neighbor data---- positive and negative ********************")
print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] Start the job")


********************* successfully get neighbor **************
********************* successfully get negative neighbor **************
********************** Successfully process neighbor data---- positive and negative ********************
[2026-02-12 18:48:11] Start the job


# Model run

In [ ]:
###########################################################
# run data
run = 0

set_random_seed(seed=run)

args.seed = run
args.save_model_name = f'{args.model_name}_seed{args.seed}'
########################################################
# set up logger
######################################

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

os.makedirs(f"{data_path}/logs/{args.model_name}/{args.dataset_name}/{args.save_model_name}/", exist_ok=True)

# create file handler that logs debug and higher level messages
fh = logging.FileHandler(f"{data_path}/logs/{args.model_name}/{args.dataset_name}/{args.save_model_name}/{str(time.time())}.log")
fh.setLevel(logging.DEBUG)

# create console handler with a higher log level
ch = logging.StreamHandler()
ch.setLevel(logging.WARNING)

# create formatter and add it to the handlers
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)

# add the handlers to logger
logger.addHandler(fh)
logger.addHandler(ch)

run_start_time = time.time()
logger.info(f"********** Run {run + 1} starts. **********")

logger.info(f'configuration is {args}')

########################################################
# create model
#####################################
dynamic_backbone = DyGMamba(node_feat_dim= 172,
                            edge_feat_dim=172,
                            time_feat_dim=args.time_feat_dim,
                            channel_embedding_dim=args.channel_embedding_dim,
                            patch_size=args.patch_size,
                            num_layers=args.num_layers,
                            num_heads=args.num_heads,
                            dropout=args.dropout,
                            gamma=args.gamma,
                            max_input_sequence_length=args.max_input_sequence_length,
                            max_interaction_times=args.max_interaction_times,
                            device=args.device)

link_predictor = MergeLayerTD(input_dim1=172,
                            input_dim2=172,
                            input_dim3=172,
                            hidden_dim=172,
                                output_dim=1)

model = nn.Sequential(dynamic_backbone, link_predictor)

logger.info(f'model -> {model}')
logger.info(f'model name: {args.model_name}, #parameters: {get_parameter_sizes(model) * 4} B, '
            f'{get_parameter_sizes(model) * 4 / 1024} KB, {get_parameter_sizes(model) * 4 / 1024 / 1024} MB.')

########################################################
# create optimizer
######################################
optimizer = create_optimizer(model=model, optimizer_name=args.optimizer,
                        learning_rate=args.learning_rate, weight_decay=args.weight_decay)

model = convert_to_gpu(model, device=args.device)

save_model_folder = f"{data_path}/saved_models/{args.model_name}/{args.dataset_name}/{args.save_model_name}"

shutil.rmtree(save_model_folder, ignore_errors=True)

os.makedirs(save_model_folder, exist_ok=True)

early_stopping = EarlyStopping(patience=args.patience, save_model_folder=save_model_folder,
                            save_model_name=args.save_model_name, logger=logger, model_name=args.model_name)

# loss_func = nn.BCELoss()
loss_func = nn.BCEWithLogitsLoss()

# ============ 新增：创建混合精度Scaler ============
scaler = GradScaler() if USE_AMP else None
if USE_AMP:
    print("混合精度训练已启用")
# =================================================

best_acc = 0

epoch = 0

########################################################
# train model
######################################

model.train()

model[0].set_neighbor_sampler(full_neighbor_sampler)

train_losses, train_metrics = [], []

idx_data_loader_tqdm = tqdm(idx_data_loader, ncols=120)
